In [8]:
"""
Convert the subsetted NextGen v2.2 hydrofabric geopackage to the partitioned model json file used in
the KF codebase.

Written by Quinn Russell
"""

import json
import geopandas as gpd

In [3]:
# # read original json format to get partitions
# with open("../1/texas_with_reservoirs_calibrated.json", "r", encoding="utf-8") as f:
#     nwm_models = json.load(f)

In [ ]:
# read gpkg to get topology, length, and geometry
gpkg = "/Users/qylee/Downloads/cat-3062147/config/cat-3062147_subset.gpkg"
flowpaths = gpd.read_file(gpkg, layer="flowpaths")
nexus = gpd.read_file(gpkg, layer="nexus")
wb_ids = gpd.read_file(gpkg, layer="id")

In [10]:
flowpaths.head()

,id,toid,mainstem,order,hydroseq,lengthkm,areasqkm,tot_drainage_areasqkm,has_divide,divide_id,poi_id,vpuid,geometry
0,wb-3062366,nex-3062126,2543533.0,1.0,15172,4.051105,6.108750,6.108750,True,cat-3062366,None,17,"MULTILINESTRING ((-1893950.163 3005214.836, -1..."
1,wb-3062446,nex-3062447,2543856.0,1.0,15171,6.814345,9.785700,9.785700,True,cat-3062446,86510,17,"MULTILINESTRING ((-1898851.096 2982419.071, -1..."
2,wb-3062447,nex-3062448,2543856.0,1.0,15170,2.593095,6.094800,15.880500,True,cat-3062447,4710,17,"MULTILINESTRING ((-1899180.632 2986582.735, -1..."
3,wb-3062456,nex-3062457,2543886.0,1.0,15169,4.061354,4.349699,4.349699,True,cat-3062456,None,17,"MULTILINESTRING ((-1895824.551 2983398.932, -1..."
4,wb-3062457,nex-3062449,2543886.0,1.0,15168,3.394750,7.897501,12.247200,True,cat-3062457,None,17,"MULTILINESTRING ((-1897560.64 2986693.115, -18..."


In [5]:
# Map NWM IDs to NextGen IDs
with open("../1/hf2.2_ref_hf_map.json", "r", encoding="utf-8") as f:
    nwm_to_ngen_mapping = json.load(f)

In [6]:
def convert_nwm_to_nextgen(reaches: list, mapping: dict) -> list:
    """Convert NWM IDs to NextGen WB IDs.

    Args:
        reaches (list): NWM reaches to convert
        mapping (dict): Mapping that looks like
            {"wb_id_1": ["nwm_id1", "nwm_id2"], "wb_id_2": ["nwm_id3", "nwm_id4", "nwm_id5], ...}

    Returns:
        list: NextGen v2.2 wb-ids
    """
    reach_floats = [float(reach) for reach in reaches]
    reverse_map = set()
    for ngen_id, nwm_ids in mapping.items():
        for nwm_id in nwm_ids:
            if nwm_id in reach_floats:
                reverse_map.add(ngen_id)

    ngen_reaches = list(reverse_map)

    wb_ids = [i.replace("cat-", "wb-", 1) if i.startswith("cat-") else i for i in ngen_reaches]

    return wb_ids

In [7]:
# This is not a particularly efficient cell but it should run in about 8 seconds

individual_models = {}
for i in [0]: # 1 models

    # sorted to improve lookup speed
    nwm_reaches = nwm_models["models"][str(i)]["model"]["reach_ids"]
    ngen_reach_ids = sorted(convert_nwm_to_nextgen(nwm_reaches, nwm_to_ngen_mapping))

    num_reaches = len(ngen_reach_ids)
    startnodes = list(range(num_reaches))
    endnodes = []

    # get endnodes
    for reach in ngen_reach_ids:
        to_nexus = flowpaths[flowpaths["id"]==reach]["toid"].values[0]
        next_reach = nexus[nexus["id"]==to_nexus]["toid"].values[0]
        if next_reach not in ngen_reach_ids:
            next_reach = reach # this is how KF handles terminal reaches
        endreach_id = ngen_reach_ids.index(next_reach)
        endnodes.append(endreach_id)

    # get lengths
    dx = []
    for reach in ngen_reach_ids:
        length = flowpaths[flowpaths["id"]==reach]["lengthkm"].values[0] * 1000 # km to m
        dx.append(length)

    # get paths
    paths = []
    for reach in ngen_reach_ids:
        linestring = flowpaths[flowpaths["id"]==reach]["geometry"].values[0]
        path = [list(pt) for pt in linestring.coords]
        paths.append(path)

    stripped_ids = [reach[3:] for reach in ngen_reach_ids]

    individual_models[str(i)] = {
        "name": str(i),
        "datetime": "2025-11-30T16:00:00+00:00",
        "timedelta": "P0DT0H15M0S",
        "reach_ids": stripped_ids,
        "startnodes": startnodes,
        "endnodes": endnodes,
        "K": [0] * num_reaches, # dummy K and x values, will be passed in from t-route
        "X": [0] * num_reaches,
        "o_t": [0.0] * num_reaches,
        "dx": dx,
        "paths": paths
    }

NameError: name 'nwm_models' is not defined

In [ ]:
all_models = {}
for i in [0]:
    all_models[str(i)] = {
        "model": individual_models[str(i)],
        "sinks": [],
        "sources": []
    }

full_partitioned_model = {
    "models": all_models,
    "connections": {}
}

In [ ]:
with open("cat-3062147.json", "w", encoding="utf-8") as model_file:
    json.dump(full_partitioned_model, model_file, indent=4)